# Call to Action (CTA) detection pipe

## 0. Task & definition

### 0.1 Label choice: binary vs. CTA types

We considered using multi-class labels (ad CTA, engagement CTA, civic CTA, etc.). but decided to go with a simpler binary label for this project:
  * `CTA` = contains at least one call to action
  * `Non-CTA` = no call to action

**Reasons for binary CTA detection:**

CTA is relatively sparse and ambiguous, so fine-grained types would require substantial training data and careful guidelines.
This will be a theme throughout but we have limited annotation capacity, fine-grained labels would be unreliable; binary detection is more robust and sufficient for downstream analysis.

---

### 0.2 Formal CTA definition grounded in literature

We will ground our definition in **marketing** + **speech-act theory**:

* In marketing, CTAs are described as **explicit prompts to get consumers to take a desired action**, such as purchasing, clicking, or signing up. Studies on CTA direct mailings and buttons treat them as messages that *urge customers to act* (e.g. “Call now”, “Click here”). [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1094996818300707?)
* In CSR social media research, CTAs are categorized as **message elements that ask audiences to engage with the cause or organization**, for example by donating, signing a petition, or sharing content. [MDPI](https://www.mdpi.com/2071-1050/13/7/3812)
* In speech-act theory, these map onto **directive speech acts**: utterances where the speaker’s aim is to get the hearer to do something in the future (requests, commands, suggestions, invitations, etc.). [ResearchGate](https://doi.org/10.15294/lc.v15i1.26029)

**Definition for the project:**

> A Call to Action (CTA) is any utterance in the transcript where a speaker **directs or encourages the listening audience to perform a specific future action**, such that the utterance’s main function is to prompt that behaviour (e.g. “visit our website”, “subscribe to the podcast”, “sign the petition in the show notes”).

* **Must be actionable**: the listener can actually do it (visit, subscribe, share, donate, sign, join, contact, etc.).
* **Targeted at listeners**, not just co-hosts (“Can you explain that again?” to the guest is *not* a CTA).
* **Includes both strong imperatives** (“Go to…”, “Use code…”) and **softer directives** (“You can support us by…”, “We’d love you to share this episode”).

**Potential issues in detecting CTAs:**

* Some instances may be borderline (e.g. rhetorical questions, vague suggestions)
* Context matters: “Check out the link in the show notes” is a CTA only if it’s prompting listener action. I.e not part of a general discussion or directed at co-hosts/guests.
* Some CTAs may be implicit or indirect.
* Some CTAs may get obscured depending on our preprocessing and whisper transcription errors.

---

## 1. Detection methods

As I see it there are three overall detection approaches we can try:

> * Rule-based detection using NLTK
> * LLM-based detection
> * Hybrid approach


### 1.1 Rule-based CTA detector

**Idea:** Use simple syntactic + lexical rules to flag candidate sentences/turns as CTAs.

**Components:**

1. **Sentence segmentation**.
2. **POS tagging / dependency parse** to find:

   * Imperative patterns (VB at sentence start, missing subject; patterns like “Please [VB] …”, “Don’t forget to [VB] …”).
   * 2nd-person subjects (“you should X”, “you can X”, “we want you to X”).
3. **CTA verb / phrase lexicon** (hand-built once, no annotation):

   * Verbs: *subscribe, follow, share, like, rate, review, visit, go to, check out, sign, donate, support, join, buy, use (code), download, click, call, email*…
   * Phrases: *“link in the show notes”*, *“use code”*, *“dot com”*, *“sign up”*, *“tell your friends”*, *“spread the word”*.
4. **Heuristics for podcast-specific ads:**

   * URLs, “dot com”, “dot org”.
   * “promo code”, “discount code”, “offer”, “free trial”.

**Rule logic (per sentence):**

* Mark as CTA if **any** of:

  1. Starts with imperative verb OR “please” + verb, AND contains 2nd person (“you”, “your”, “listeners”) or implicit “you” (no subject).
  2. Contains CTA verb from lexicon AND “you/your/guys/everyone/listeners”.
  3. Contains URL-like token or words like “use code”, “visit dot”.
  4. Contains patterns like “don’t forget to”, “make sure to”, “be sure to”.

We can **aggregate to turn level**: a turn is CTA-positive if any of its sentences are positive. Possibly keeping track of the number of CTA sentences per turn.

### 1.2 LLM-based detector

Optimally we would fine-tune a small classifier using a pre-trained model like BERT or RoBERTa on a small labeled set, but since we don't have recources to do this, we will use a causal language model to leverage semantic understanding.

Here we may use a few-hot based approach on a sentance level, with aggregation to turn level as above.

### 1.3 Hybrid model (LLM + rules)

**Design options to think about:**

We could perfom both of the above methods and combine their outputs in different ways:

1. **High-precision hybrid (intersection):**

   * Predict CTA if **both** rule-based detector and LLM say CTA.
   * Use this for conservative counts or qualitative examples.
2. **High-recall hybrid (union):**

   * Predict CTA if **either** rule-based or LLM says CTA.
   * Use this for exploratory analyses where missing CTAs is worse than a few extra false positives.
3. **Tiered confidence:**

   * `High-confidence CTA`: rule + LLM both CTA.
   * `Possible CTA`: only one method says CTA.
   * `Non-CTA`: both say non-CTA.

Alternatively, we could use the rule-based detector to flag candidates, and then have the LLM re-classify only those candidates to filter false positives. This would reduce LLM calls but might miss CTAs that rules don’t catch.
For now I've looked at using "microsoft/Phi-3-instruct" but it is too demanding to run on the whole dataset, so we can either try a smaller model/ Upgrade the hardware or just use the hybrid approach as described above.

---

## 2. Text processing approaches

### 2.1 Units to consider

1. **Turn-level analysis**

   * Input: entire speaker turn text.
   * Label: “contains CTA or not”.

   **Pros:** aligns with conversational structure; easy to link to speaker role, interruptions.
   **Cons:** long turns can contain mixed content; CTAs may be short snippets.

2. **Sentence-based analysis**

   * Split transcript or turns into sentences; classify each sentence.
   * Aggregate back to turn or episode (e.g., a turn is CTA if any sentence is CTA).

   **Pros:** more precise localization of CTAs; easier for rule-based patterns (imperatives). May be better for LLMs which have context length limits. If we do it on transcripts we can access more data points for evaluation.
   **Cons:** sentence splitting on transcripts is imperfect; slightly more bookkeeping. If we do it on turns we may miss cross-sentence CTAs.


### 2.2 What I've done for now

I've implimented both detection approaches for each turn on a sentance level. Since i think sentence-level gives better alignment with the literature’s notion of CTAs as discrete “prompts” or “utterances”, and doing it this way allows us to keep the turn level metadata (speaker role, episode ID) for aggregation and analysis.

I've also done some cleaning of the transcripts to remove non-speech elements like [music], as well as removing very short sentences that are unlikely to contain CTAs.

---

## 4. Summary of design decisions

1. **Conceptually considered**:

   * Binary vs multi-type CTA labels; chose binary for reliability under limited annotation resources.
   * Turn-level vs sentence-level vs sliding window units; chose sentence-level + turn aggregation as a compromise between precision and practicality.
2. **Methodologically compared**:

   * Rule-based imperative / lexicon detection vs LLM-based zero/few-shot classification vs hybrid.
   * Different confidence regimes (intersection vs union of methods).



In [1]:
from helpers.sporc import SPORCDataset
import pandas as pd

sporc = SPORCDataset(local_data_dir="data", load_samples_only=True, show_progress=False)
episodes = sporc.get_all_episodes()

episodes_df = pd.read_parquet("data/episodes.parquet")
turns_df = pd.read_parquet("data/turns.parquet")

episode_lookup = {ep.mp3_url: ep for ep in episodes}
episodes = (episodes_df['mp3_url'].map(episode_lookup)).to_list()

INFO:helpers.sporc.dataset:Loading SPORC dataset from local directory: data
INFO:helpers.sporc.dataset:Loading SPORC sample dataset only.
INFO:helpers.sporc.dataset:✓ All required files found
INFO:helpers.sporc.dataset:Loading all records from local files into memory...
INFO:helpers.sporc.dataset:Loading episode_data_sample...
INFO:helpers.sporc.dataset:Loading speaker_turn_data_sample...
INFO:helpers.sporc.dataset:✓ Loaded 210,000 total records from 2 files
INFO:helpers.sporc.dataset:✓ Local dataset loaded successfully in 6.57 seconds
INFO:helpers.sporc.dataset:✓ Dataset loaded successfully with 210000 total records
INFO:helpers.sporc.dataset:Processing dataset into Podcast and Episode objects...
INFO:helpers.sporc.dataset:Separating episode data from speaker turn data...
INFO:helpers.sporc.dataset:✓ Separation completed in 0.15 seconds
INFO:helpers.sporc.dataset:  Episode records: 10,000, Speaker turn records: 200,000
INFO:helpers.sporc.dataset:Grouping episodes by podcast...
INFO:he

### Per sentence splitting and cleaning

In [3]:
import re
import pandas as pd

sentence_splitter = re.compile(r"(?<=[.!?])\s+")

MUSIC_PATTERN = re.compile(
    r"""
    # Proper bracketed versions
    (\[\s*music\s*\]|\(\s*music\s*\))
    |
    # Broken left bracket versions: [Music, [music, [ MUSIC
    (\[\s*music)
    |
    # Broken right bracket versions: Music], music], MUSIC]
    (music\s*\])
    |
    # Standalone 'music' when used as a tag (not sentence content)
    ^\s*music\s*$ 
    """,
    re.IGNORECASE | re.VERBOSE
)


def is_stump(sentence):
    # Remove punctuation-only sentences, single-word fragments, etc.
    cleaned = re.sub(r'[^\w\s]', '', sentence).strip()
    return len(cleaned.split()) <= 1

def is_mostly_brackets(sentence):
    # Detect chains like "[Music] [Music] [Music]"
    bracket_chars = sum(c in "[]()" for c in sentence)
    return bracket_chars / max(len(sentence), 1) > 0.4

def clean_sentence(sentence):
    # Remove any explicit or broken music tags
    s = MUSIC_PATTERN.sub('', sentence).strip()

    # Remove leftover bracket debris
    s = re.sub(r'[\[\]\(\)"]+', '', s).strip()

    return s


def text_to_sentences(text):
    raw_sentences = sentence_splitter.split(text)
    cleaned = []
    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue

        s = clean_sentence(s)
        if not s:
            continue

        if is_stump(s):
            continue

        if is_mostly_brackets(s):
            continue

        cleaned.append(s)
    return cleaned

In [4]:
ep = episodes[2]

# make df of sentences from transcript with episode name

sentences_transcript = text_to_sentences(ep.transcript)
sentences_transcript 

["I'm Simon Shapiro and this is Sing Out Speak Out.",
 "Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me.",
 "My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out.",
 'Well hello everyone and welcome to Sing Out Speak Out episode number 20.',
 "I know crazy, it's going fast this crazy year that we are having 2020.",
 "I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever.",
 "But I know it

In [5]:
rows = []
    
for idx, ep in enumerate(episodes[:10]):  # limit for testing
    rows.append({
        "idx": idx,
        "title": getattr(ep, "title", None),
        "podcast_title": getattr(ep, "podcast_title", None),
        "sentences": text_to_sentences(getattr(ep, "transcript", "")),
    })


df_transcript = pd.DataFrame(rows)

df_transcript = df_transcript.explode("sentences").rename(columns={"sentences": "text"}).reset_index(drop=True)
df_transcript["text"] = df_transcript["text"].astype("string")

In [6]:
df_transcript

,idx,title,podcast_title,text
0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.
1,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Here I give you the philosophies, ideas and so..."
2,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,My will to connect with you and share the best...
3,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,So here are my five F's of being five.
4,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Kids are amazing at jumping into fantasy, not ..."
...,...,...,...,...
1909,9,What causes anxiety?,Anxiety 2 Confidence podcast,I'm always happy to answer any questions that ...
1910,9,What causes anxiety?,Anxiety 2 Confidence podcast,I hope you have a safe week and until next wee...
1911,9,What causes anxiety?,Anxiety 2 Confidence podcast,Thank you for listening to this episode of the...
1912,9,What causes anxiety?,Anxiety 2 Confidence podcast,You can find more information and my extensive...


In [7]:
sentences_turns = []

ep = episodes[2]
turns = ep.get_all_turns()
for turn in turns:
    sentences_turns.extend(text_to_sentences(turn.text))

sentences_turns

["I'm Simon Shapiro and this is Sing Out Speak Out.",
 "Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me.",
 "My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out",
 'Well hello everyone and welcome',
 'to Sing Out Speak Out episode number 20.',
 "I know crazy, it's going fast this crazy year that we are having 2020.",
 "I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever.",
 "But I know

In [8]:
rows = []

for ep_idx, ep in enumerate(episodes[:10]):  # or all episodes
    turns = ep.get_all_turns()

    for turn_idx, turn in enumerate(turns):
        turn_text = getattr(turn, "text", None)

        # Skip empty/None turns
        if not turn_text:
            continue

        rows.append({
            "episode_idx": ep_idx,
            "turn_idx": turn_idx,
            "title": getattr(ep, "title", None),
            "podcast_title": getattr(ep, "podcast_title", None),
            "turn_text": turn_text,
            "sentences": text_to_sentences(turn_text)
        })

df_turns = pd.DataFrame(rows)

df_turns_sentences = (
    df_turns
    .explode("sentences")
    .rename(columns={"sentences": "text"})
    .reset_index(drop=True)
)
df_turns_sentences["text"] = df_turns_sentences["text"].astype("string")

In [9]:
df_turns_sentences[df_turns_sentences['episode_idx'] == 2]

,episode_idx,turn_idx,title,podcast_title,turn_text,text
18,2,0,Today Is Yesterday,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak Out.
19,2,0,Today Is Yesterday,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,"Here I give you the philosophies, ideas and so..."
20,2,0,Today Is Yesterday,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,My will to connect with you and share the best...
21,2,1,Today Is Yesterday,SingOut SpeakOut,.,<NA>
22,2,2,Today Is Yesterday,SingOut SpeakOut,Well hello everyone and welcome,Well hello everyone and welcome
23,2,3,Today Is Yesterday,SingOut SpeakOut,to Sing Out Speak Out episode number 20. I kn...,to Sing Out Speak Out episode number 20.
24,2,3,Today Is Yesterday,SingOut SpeakOut,to Sing Out Speak Out episode number 20. I kn...,"I know crazy, it's going fast this crazy year ..."
25,2,3,Today Is Yesterday,SingOut SpeakOut,to Sing Out Speak Out episode number 20. I kn...,I know for a lot of people the hard lockdown a...
26,2,3,Today Is Yesterday,SingOut SpeakOut,to Sing Out Speak Out episode number 20. I kn...,But I know it can be tough when you're stuck i...
27,2,3,Today Is Yesterday,SingOut SpeakOut,to Sing Out Speak Out episode number 20. I kn...,Every day is a bit the same.


In [10]:
raw = []

turns = episodes[2].get_all_turns()
for turn in turns:
    raw.append(turn.text)

raw

[" I'm Simon Shapiro and this is Sing Out Speak Out. Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me. My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out",
 '.',
 ' Well hello everyone and welcome',
 " to Sing Out Speak Out episode number 20. I know crazy, it's going fast this crazy year that we are having 2020. I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever. But I know it can be 

### Per turn cleaning

In [11]:
# Remove any leftover bracket debris
BRACKET_JUNK = re.compile(r'[\[\]\(\)"“”]+')
def clean_turn(text):
    if not text or not text.strip():
        return ""
    
    # Step 1 — Remove music-tag patterns
    cleaned = MUSIC_PATTERN.sub(" ", text)
    
    # Step 2 — Remove leftover bracket garbage ("[", "]", "[", etc.)
    cleaned = BRACKET_JUNK.sub(" ", cleaned)
    
    # Step 3 — Normalize spacing
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # Step 4 — Drop stump turns (no words)
    if not re.search(r'\w', cleaned):
        return ""
    
    # Step 5 — Optional: Drop short meaningless junk (< 5 characters)
    if len(cleaned) < 5:
        return ""
    
    return cleaned

In [12]:
ep = episodes[2]
cleaned_turns = [
    cleaned for cleaned in (clean_turn(turn.text) for turn in ep.get_all_turns())
    if cleaned
]
cleaned_turns

["I'm Simon Shapiro and this is Sing Out Speak Out. Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me. My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out",
 'Well hello everyone and welcome',
 "to Sing Out Speak Out episode number 20. I know crazy, it's going fast this crazy year that we are having 2020. I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever. But I know it can be tough whe

## Rules-based CTA detection

In [13]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
from nltk import word_tokenize, pos_tag
from typing import List


# Basic CTA lexicon (test)
CTA_VERBS = {
    "subscribe", "follow", "share", "like", "rate", "review", "visit", "go", "check", "sign", "donate",
    "support", "join", "buy", "use", "download", "click", "call", "email", "register", "sign up",
    "tell", "spread"
}

CTA_PHRASES = {
    "use code", "promo code", "discount code", "free trial", "link in the show notes", "in the show notes",
    "show notes", "dot com", "dot org", "our website", "visit us at"
}

URL_PATTERN = re.compile(r"https?://|www\\.|\\.com|\\.org|\\.net", re.IGNORECASE)


def contains_url_like(text: str) -> bool:
    return bool(URL_PATTERN.search(text))


def contains_cta_phrase(text: str) -> bool:
    lowered = text.lower()
    return any(phrase in lowered for phrase in CTA_PHRASES)


def tokens_and_pos(sentence: str):
    tokens = word_tokenize(sentence)
    pos = pos_tag(tokens)
    return tokens, pos


def is_imperative(tokens: List[str], pos: List[tuple]) -> bool:
    """Very simple imperative heuristic:
    - Sentence starts with a base form verb (VB) or 'please' followed by VB.
    - We do not require an explicit subject ('you').
    """
    if not tokens:
        return False

    # Case 1: starts with a verb in base form
    first_word, first_tag = pos[0]
    if first_tag == "VB":
        return True

    # Case 2: starts with 'please' + verb
    if tokens[0].lower() == "please" and len(pos) > 1 and pos[1][1] == "VB":
        return True

    return False


def has_second_person(tokens: List[str]) -> bool:
    second_person = {"you", "your", "yours", "y'all"}
    return any(t.lower() in second_person for t in tokens)


def has_cta_lexicon(tokens: List[str]) -> bool:
    lowered = [t.lower() for t in tokens]
    # Allow for multi-word cues like 'sign up'
    text = " ".join(lowered)
    if any(phrase in text for phrase in CTA_VERBS if " " in phrase):
        return True

    # Single-token verbs
    return any(t in CTA_VERBS for t in lowered)


def rule_based_cta_label(sentence: str) -> int:
    """Return 1 if sentence is classified as CTA by rules, else 0.

    Rules (any of these is sufficient):
    - Imperative structure with 2nd person or CTA lexicon.
    - Contains CTA lexicon with explicit 'you' or similar.
    - Contains URL-like token or CTA-specific phrase (promo codes, show notes).
    """
    if not isinstance(sentence, str):
        return pd.NA

    text = sentence.strip()
    if not text:
        return 0

    tokens, pos_tags = tokens_and_pos(text)

    # High-salience patterns
    if contains_url_like(text) or contains_cta_phrase(text):
        return 1

    # Imperative with CTA lexicon or 2nd person
    if is_imperative(tokens, pos_tags) and (has_cta_lexicon(tokens) or has_second_person(tokens)):
        return 1

    # Non-imperative, but CTA verb + second person
    if has_cta_lexicon(tokens) and has_second_person(tokens):
        return 1

    return 0


In [ ]:
# We test this with sentences from cleaned_turns
cta_labels = [rule_based_cta_label(s) for s in cleaned_turns]
# print coloured accordingly
for s, label in zip(cleaned_turns, cta_labels):
    if label == 1:
        print(f"\033[92m{s}\033[0m")  # Green for CTA
    else:
        print(f"\033[91m{s}\033[0m")  # Red for non-CTA

I'm Simon Shapiro and this is Sing Out Speak Out. Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me. My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out
Well hello everyone and welcome
to Sing Out Speak Out episode number 20. I know crazy, it's going fast this crazy year that we are having 2020. I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever. But I know it can be tough when you're s

In [ ]:
cta_labels = [rule_based_cta_label(s) for s in sentences_turns]

for s, label in zip(sentences_turns, cta_labels):
    if label == 1:
        print(f"\033[92m{s}\033[0m")  # Green for CTA
    else:
        print(f"\033[91m{s}\033[0m")  # Red for non-CTA

I'm Simon Shapiro and this is Sing Out Speak Out.
Here I give you the philosophies, ideas and songs I've been working on for many years but until now I've shared just too few of because I've been afraid of you and worse I've been afraid of me.
My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out
Well hello everyone and welcome
to Sing Out Speak Out episode number 20.
I know crazy, it's going fast this crazy year that we are having 2020.
I know for a lot of people the hard lockdown and the restrictions due to COVID-19 are still in full force and in Australia we are very lucky that we've managed to squash that curve for now due I think in part because we have a smaller population and we had advanced warning of what was happening in other countries so I think that held us in good stead but we're not over it yet of course and we're not gloating in any way whatsoever.
But I know it can be tough when you're s

Only the last sentence actually contains a CTA in my opinion.

we'll try to apply to the dataframes

In [ ]:
# apply rule_based_cta_label to df_transcript and df_turns_sentences

df_transcript['cta_label'] = df_transcript['text'].apply(rule_based_cta_label)
df_turns_sentences['cta_label'] = df_turns_sentences['text'].apply(rule_based_cta_label)

In [ ]:
df_transcript.head()

,idx,title,podcast_title,text,cta_label
0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.,0
1,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Here I give you the philosophies, ideas and so...",0
2,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,My will to connect with you and share the best...,1
3,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,So here are my five F's of being five.,0
4,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Kids are amazing at jumping into fantasy, not ...",0


In [ ]:
df_turns_sentences.head()

,episode_idx,turn_idx,title,podcast_title,turn_text,text,cta_label
0,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak Out.,0
1,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,"Here I give you the philosophies, ideas and so...",0
2,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,My will to connect with you and share the best...,1
3,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,So here are my five F's of being five.,0
4,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak ...,"Kids are amazing at jumping into fantasy, not ...",0


## LLM-based CTA detection

In [2]:
from typing import Dict
import time
import json
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
)
import torch
torch.__version__

'2.5.1'

In [ ]:
def load_llm(model_name: str = "microsoft/Phi-3-mini-128k-instruct",
             device: str = "cpu"):
    """
    Load a Hugging Face pipeline.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    llm = pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=50,
        do_sample=False
    )
    return llm, tokenizer


In [ ]:
llm, tokenizer = load_llm()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

c:\Users\au700932\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\au700932\.cache\huggingface\hub\models--microsoft--Phi-3-mini-128k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:  25%|##4       | 661M/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:  16%|#6        | 807M/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [4]:
PHI_3_SYSTEM_PROMPT_0 = (
    "You are a text classification model. "
    "Your task is to determine whether a given text contains a call-to-action (CTA). "
    "A CTA is any direct or indirect attempt to persuade the audience to take a specific future action. "
    "Respond only with 'YES' or 'NO'."
)

PHI_3_SYSTEM_PROMPT_1 = (
    "You are a text classification model that determines whether a given sentence contains a Call to Action (CTA). "
    "A CTA is any utterance where the speaker urges, requests, or encourages the audience to take a specific action. "
    "Classify each example strictly as 'YES' for CTA or 'NO' for Non-CTA. "
    "Provide no additional explanation unless explicitly asked."
)


# few-shot pairs
FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Text: If you enjoy this podcast, please subscribe and leave us a review.\nDoes this contain a CTA?"
    },
    {
        "role": "assistant",
        "content": "YES"
    },
    {
        "role": "user",
        "content": "Text: Climate change is a complex issue with many drivers.\nDoes this contain a CTA?"
    },
    {
        "role": "assistant",
        "content": "NO"
    },
    {
        "role": "user",
        "content": "Text: There is a link in the show notes if you want to support the campaign.\nDoes this contain a CTA?"
    },
    {
        "role": "assistant",
        "content": "YES"
    },
    {
        "role": "user",
        "content": "Text: Thanks for listening, see you next time.\nDoes this contain a CTA?"
    },
    {
        "role": "assistant",
        "content": "NO"
    },
]

def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": PHI_3_SYSTEM_PROMPT_1},
        *FEW_SHOT_EXAMPLES,
        {
            "role": "user",
            "content": (
                f"Text: {text}\n"
                "Does this contain a CTA?"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [5]:
print(build_prompt("Please subscribe for more updates."))

<|system|>
You are a text classification model that determines whether a given sentence contains a Call to Action (CTA). A CTA is any utterance where the speaker urges, requests, or encourages the audience to take a specific action. Classify each example strictly as 'YES' for CTA or 'NO' for Non-CTA. Provide no additional explanation unless explicitly asked.<|end|>
<|user|>
Text: If you enjoy this podcast, please subscribe and leave us a review.
Does this contain a CTA?<|end|>
<|assistant|>
YES<|end|>
<|user|>
Text: Climate change is a complex issue with many drivers.
Does this contain a CTA?<|end|>
<|assistant|>
NO<|end|>
<|user|>
Text: There is a link in the show notes if you want to support the campaign.
Does this contain a CTA?<|end|>
<|assistant|>
YES<|end|>
<|user|>
Text: Thanks for listening, see you next time.
Does this contain a CTA?<|end|>
<|assistant|>
NO<|end|>
<|user|>
Text: Please subscribe for more updates.
Does this contain a CTA?<|end|>
<|assistant|>



In [ ]:
prompt = build_prompt("Please subscribe to my channel for more updates.")
output = llm(prompt, max_new_tokens=5, temperature=0.0, return_full_text=False)[0]["generated_text"]
print(output)

 YES


## FULL ON DETECTION

In [ ]:
rows = []

for ep_idx, ep in enumerate(episodes):
    turns = ep.get_all_turns()

    for turn_idx, turn in enumerate(turns):
        turn_text = getattr(turn, "text", None)

        # Skip empty/None turns
        if not turn_text:
            continue

        rows.append({
            "episode_idx": ep_idx,
            "turn_idx": turn_idx,
            "title": getattr(ep, "title", None),
            "podcast_title": getattr(ep, "podcast_title", None),
            "sentences": text_to_sentences(turn_text),
            
        })

df_turns = pd.DataFrame(rows)

df_turns_sentences = (
    df_turns
    .explode("sentences")
    .rename(columns={"sentences": "text"})
    .reset_index(drop=True)
)
df_turns_sentences["text"] = df_turns_sentences["text"].astype("string")

In [ ]:
df_turns_sentences

,episode_idx,turn_idx,title,podcast_title,text
0,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.
1,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Here I give you the philosophies, ideas and so..."
2,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,My will to connect with you and share the best...
3,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,So here are my five F's of being five.
4,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Kids are amazing at jumping into fantasy, not ..."
...,...,...,...,...,...
504792,1167,113,072 Acts 2 Distinctives Series- the Importance...,Strength for Today's Pastor,And it really locks in their ability to share ...
504793,1167,113,072 Acts 2 Distinctives Series- the Importance...,Strength for Today's Pastor,It really locks in the fundamentals of how the...
504794,1167,113,072 Acts 2 Distinctives Series- the Importance...,Strength for Today's Pastor,And then we will usually video most if not all...
504795,1167,113,072 Acts 2 Distinctives Series- the Importance...,Strength for Today's Pastor,Because it gives us the sense these people are...


In [ ]:
df_turns_sentences['cta_label'] = df_turns_sentences['text'].apply(rule_based_cta_label)

In [ ]:
df_turns_sentences[df_turns_sentences['cta_label'] == 1]['text']

2         My will to connect with you and share the best...
9         My will to connect with you and share the best...
14        I hope you like it and I hope you have a fanta...
20        My will to connect with you and share the best...
26        But I know it can be tough when you're stuck i...
                                ...                        
504687    Every one of you in the name of Jesus Christ, ...
504704               I'll tell you that, to cut through the
504707    Yeah, and I've heard stories in the Scandinavi...
504747    And I love that whole mikvah picture because y...
504762    I think often, and like I said, that could be,...
Name: text, Length: 37575, dtype: string

In [ ]:
df_turns_sentences.to_csv("data/turns_sentences_cta_labels.csv", index=False)

In [ ]:
df_turns_sentences = pd.read_csv("data/turns_sentences_cta_labels.csv")
df_turns_sentences.head()

,episode_idx,turn_idx,title,podcast_title,text,cta_label
0,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.,0.0
1,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Here I give you the philosophies, ideas and so...",0.0
2,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,My will to connect with you and share the best...,1.0
3,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,So here are my five F's of being five.,0.0
4,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Kids are amazing at jumping into fantasy, not ...",0.0


### LLM

In [ ]:
time_per_sample_sec= 19 / 24  # approx time per sample in seconds
n_valid_samples = df_turns_sentences[~df_turns_sentences['text'].isna()].shape[0]
print(f"Approximate total time to process all valid samples on T4 using pipeline: {n_valid_samples * time_per_sample_sec / 60 / 60: .2f} hours")

Approximate total time to process all valid samples on T4 using pipeline:  95.73 hours


In [ ]:
time_per_sample_sec= 18 / 24  # approx time per sample in seconds
n_valid_samples = df_turns_sentences[~df_turns_sentences['text'].isna()].shape[0]
print(f"Approximate total time to process all valid samples on T4 using pipeline + quantization: {n_valid_samples * time_per_sample_sec / 60 / 60: .2f} hours")

Approximate total time to process all valid samples on T4 using pipeline + quantization:  90.69 hours


In [ ]:
time_per_sample_sec= 19 / 24  # approx time per sample in seconds
n_flagged_samples = df_turns_sentences[df_turns_sentences['cta_label'] == 1].shape[0]
print(f"Approximate total time to process all valid samples on T4 using pipeline: {n_flagged_samples * time_per_sample_sec / 60 / 60: .2f} hours")

Approximate total time to process all valid samples on T4 using pipeline:  8.26 hours


### considerations:
* Smaller model for faster processing time, at the cost of some accuracy.
* Using the models directly without pipeline + batching. (tried on the 24 samples and it was not faster)
* Change the way i build prompts using prefixes so i dont have to include the whole system prompt every time.
* GGUF and run on multicore cluster.
* Fine-tune classifier on small golden set??

## Downstream analysis

In [ ]:
df_turns_sentences = pd.read_csv("data/turns_sentences_cta_labels.csv")

print("Columns:", df_turns_sentences.columns.tolist())
print("\nHead:")
display(df_turns_sentences.head())

# Basic label distribution at sentence level
print("\nSentence-level CTA label counts:")
print(df_turns_sentences["cta_label"].value_counts())

print("\nSentence-level CTA label proportions:")
print(df_turns_sentences["cta_label"].value_counts(normalize=True))

print("\nShare of sentences that are CTA:")
print(df_turns_sentences["cta_label"].mean())

Columns: ['episode_idx', 'turn_idx', 'title', 'podcast_title', 'text', 'cta_label']

Head:


,episode_idx,turn_idx,title,podcast_title,text,cta_label
0,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.,0.0
1,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Here I give you the philosophies, ideas and so...",0.0
2,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,My will to connect with you and share the best...,1.0
3,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,So here are my five F's of being five.,0.0
4,0,0,Best of SingOut SpeakOut No.3,SingOut SpeakOut,"Kids are amazing at jumping into fantasy, not ...",0.0



Sentence-level CTA label counts:
cta_label
0.0    397743
1.0     37575
Name: count, dtype: int64

Sentence-level CTA label proportions:
cta_label
0.0    0.913684
1.0    0.086316
Name: proportion, dtype: float64

Share of sentences that are CTA:
0.08631621021873664


In [ ]:
# Aggregate to turn level: whether a turn contains at least one CTA sentence
turn_level = (
    df_turns_sentences
    .groupby(["episode_idx", "turn_idx"])
    .agg(
        has_cta=("cta_label", "max"),   # 1 if any sentence is CTA
        n_sentences=("text", "size")
    )
    .reset_index()
)

turn_level["has_cta"] = turn_level["has_cta"].astype(bool)

print("Turn-level summary (first 10 rows):")
display(turn_level.head(10))

print("\nShare of turns that contain at least one CTA:")
print(turn_level["has_cta"].mean())

print("\nDistribution of number of sentences per turn:")
print(turn_level["n_sentences"].describe())


Turn-level summary (first 10 rows):


,episode_idx,turn_idx,has_cta,n_sentences
0,0,0,True,7
1,1,0,True,9
2,1,1,True,1
3,1,2,True,1
4,2,0,True,3
5,2,1,True,1
6,2,2,False,1
7,2,3,True,16
8,2,4,True,1
9,2,5,True,1



Share of turns that contain at least one CTA:
0.4601310334802071

Distribution of number of sentences per turn:
count    199491.000000
mean          2.530425
std          13.932043
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2639.000000
Name: n_sentences, dtype: float64


In [ ]:
df_turns_sentences[df_turns_sentences["episode_idx"]==1]

,episode_idx,turn_idx,title,podcast_title,text,cta_label
7,1,0,It's All Gone,SingOut SpeakOut,I'm Simon Shapiro and this is Sing Out Speak Out.,0.0
8,1,0,It's All Gone,SingOut SpeakOut,"Here I give you the philosophies, ideas and so...",0.0
9,1,0,It's All Gone,SingOut SpeakOut,My will to connect with you and share the best...,1.0
10,1,0,It's All Gone,SingOut SpeakOut,Well everyone it's been a crazy busy week and ...,0.0
11,1,0,It's All Gone,SingOut SpeakOut,Apologies it's very late.,0.0
12,1,0,It's All Gone,SingOut SpeakOut,As I've been chasing my tail the whole weekend...,0.0
13,1,0,It's All Gone,SingOut SpeakOut,It's all gone.,0.0
14,1,0,It's All Gone,SingOut SpeakOut,I hope you like it and I hope you have a fanta...,1.0
15,1,0,It's All Gone,SingOut SpeakOut,I'll be back next week with another song and a...,0.0
16,1,1,It's All Gone,SingOut SpeakOut,NaN,NaN


In [ ]:
# Episode metadata (title, podcast_title) – just take the first occurrence per episode
episode_meta = (
    df_turns_sentences
    .groupby("episode_idx")[["title", "podcast_title"]]
    .agg(lambda x: x.iloc[0])
)

# Episode-level CTA stats (using turn-level has_cta)
episode_cta = (
    turn_level
    .groupby("episode_idx")["has_cta"]
    .agg(
        n_cta_turns="sum",
        n_turns="size"
    )
)

episode_cta["cta_turn_share"] = episode_cta["n_cta_turns"] / episode_cta["n_turns"]

episode_summary = (
    episode_cta
    .join(episode_meta, how="left")
    .reset_index()
)

print("Episode-level CTA summary (sorted by CTA turn share):")
display(
    episode_summary
    .sort_values("cta_turn_share", ascending=False)
    .head(20)
)

print("\nOverall mean share of turns with CTA per episode:")
print(episode_summary["cta_turn_share"].mean())


Episode-level CTA summary (sorted by CTA turn share):


,episode_idx,n_cta_turns,n_turns,cta_turn_share,title,podcast_title
0,0,1,1,1.0,Best of SingOut SpeakOut No.3,SingOut SpeakOut
197,324,2,2,1.0,Peaceful Place Relaxation Recording – Episode 228,Ted in Your Head
205,332,3,3,1.0,Friday FAQ: How Many Hypnotherapy Sessions Wil...,Ted in Your Head
204,331,3,3,1.0,Thoughts on Grief and Loss – Episode 223,Ted in Your Head
201,328,5,5,1.0,Friday FAQ: Will You Make Me Cluck Like a Chic...,Ted in Your Head
718,851,1,1,1.0,Magical Morning affirmations,Daily cup of Prana “ Guided Meditations & Insp...
199,326,2,2,1.0,Peaceful Place Relaxation Recording – Episode 228,Ted in Your Head
198,325,3,3,1.0,Meditation Revisited – Episode 229,Ted in Your Head
196,323,5,5,1.0,Friday FAQ: Can Hypnotherapy Help Me Sleep Bet...,Ted in Your Head
754,887,4,4,1.0,057: How To Build Your Online Personal Brand E...,Growth Strategist Podcast With Wilson Komala (...



Overall mean share of turns with CTA per episode:
0.5958274566342815


In [ ]:
# Sentence-level CTA by show
show_sentence_cta = (
    df_turns_sentences
    .groupby("podcast_title")["cta_label"]
    .agg(
        n_cta_sentences="sum",
        n_sentences="size"
    )
)

show_sentence_cta["cta_sentence_share"] = (
    show_sentence_cta["n_cta_sentences"] / show_sentence_cta["n_sentences"]
)

print("CTA at sentence level by show:")
display(
    show_sentence_cta
    .sort_values("cta_sentence_share", ascending=False)
    .head(20)
)

# Turn-level CTA by show for a slightly different view
show_turn_cta = (
    turn_level
    .merge(
        episode_meta.reset_index()[["episode_idx", "podcast_title"]],
        on="episode_idx",
        how="left"
    )
    .groupby("podcast_title")["has_cta"]
    .agg(
        n_cta_turns="sum",
        n_turns="size"
    )
)

show_turn_cta["cta_turn_share"] = show_turn_cta["n_cta_turns"] / show_turn_cta["n_turns"]


CTA at sentence level by show:


,n_cta_sentences,n_sentences,cta_sentence_share
podcast_title,,,
Let’s Be Vocal,33.0,87,0.379310
Office Tech EDU,227.0,655,0.346565
Hear My Voice,2.0,8,0.250000
Bob Dylan: Album By Album,23.0,92,0.250000
Amplified Podcast,39.0,173,0.225434
We Love Tech Podcast,48.0,236,0.203390
Privateer Rum,736.0,3761,0.195693
Vegan Bites With Nekei,33.0,178,0.185393
Theme Park Hipster,143.0,772,0.185233


In [ ]:
cta_by_cat = usable_df.groupby("main_category")["has_cta"].mean().sort_index()
print("\nCTA share by category:")
display(cta_by_cat)

#### 1. Detection Methods to consider:
- Rule-based detection using NLTK for imperative verbs and command structures
- LLM-based detection using zero/few-shot prompting
- Hybrid approach combining both methods

#### 2. Text Processing Approaches:
- Turn-level analysis: Analyze each conversation turn independently
    - Pros: Natural conversation boundaries, speaker context
    - Cons: Some episodes lack turn annotations, turns can be imperfect
- Sliding window analysis: Process transcript with overlapping chunks
    - Pros: Handles episodes without turns, maintains context
    - Cons: May split CTAs across chunks, loses speaker information

#### 3. Possible Evaluation Strategy:
- Manual annotation of sample data for benchmarking
- Compare agreement between rule-based and LLM approaches
- Analyze false positives/negatives to refine detection
- Test robustness across different podcast genres

#### 4. Analysis Plan:
- CTA frequency across genres and episode types
- Temporal distribution within episodes
- Speaker analysis (host vs guest CTAs)
- Common CTA types and patterns
